In [2]:
import math
import random
import copy
import sys
import os

# ==============================================================================
# CẤU HÌNH HỆ THỐNG & ĐẦU VÀO / ĐẦU RA
# ==============================================================================
USE_EXCEL = 1           # 0: Dùng dữ liệu mẫu trong code, 1: Đọc/Ghi file Excel
IS_PEAK_HOUR = True     # True: Giờ cao điểm (J=1), False: Giờ bình thường (J=2)

INPUT_EXCEL_FILE = "input.xlsx"
OUTPUT_EXCEL_FILE = "output_schedule.xlsx"

# Thời gian lăn trung bình từ Gate ra điểm chờ Runway theo từng Nhà ga (giây)
TERMINAL_TAXI_TIME = {
    'T1': 6 * 60,   # 360s (6 phút)
    'T2': 4 * 60,   # 240s (4 phút)
    'T3': 8 * 60    # 480s (8 phút)
}

MAX_GATE_HOLD_TIME = 600    # Thời gian tối đa được giữ máy bay tại Gate (giây)
CROSSING_TAXI_TIME = 0      # Biến Y: Thời gian Arrival lăn từ lúc hạ cánh X tới điểm cắt ngang đường băng (giây) - Mặc định = 0 (bản gốc là 105s)
NUM_RUNWAYS = 1             # Mặc định hệ thống có 1 đường băng duy nhất

# ==============================================================================
# THAM SỐ THUẬT TOÁN SA & LNS
# ==============================================================================
NUM_SALNS_RUNS = 10         # Số lần chạy lặp lại SALNS để chọn nghiệm tốt nhất
T_0 = 1000.0                # Nhiệt độ ban đầu
T_L = 1.0                   # Nhiệt độ kết thúc
LAMBDA = 0.90               # Hệ số làm lạnh
BETA = 50                   # Số vòng lặp ở mỗi mức nhiệt độ
THETA_NOTIMP = 20           # Số mức nhiệt độ liên tiếp không cải thiện để dừng

# ==============================================================================
# MA TRẬN KHOẢNG CÁCH GIÃN CÁCH (SEPARATION MATRIX) - Đơn vị: Giây
# Hàng: Máy bay đi trước (Leader) | Cột: Máy bay đi sau (Follower)
# Hạng: S (Super), H (Heavy), M (Medium), L (Light) cho Departure.
# Hạng A: Arrival khi cắt ngang đường băng (chặn 30s cho mọi máy bay sau).
# ==============================================================================
SEPARATION_MATRIX = {
    'S': {'S': 90,  'H': 120, 'M': 150, 'L': 180, 'A': 60},
    'H': {'S': 90,  'H': 90,  'M': 120, 'L': 150, 'A': 60},
    'M': {'S': 60,  'H': 60,  'M': 60,  'L': 90,  'A': 60},
    'L': {'S': 60,  'H': 60,  'M': 60,  'L': 60,  'A': 60},
    'A': {'S': 30,  'H': 30,  'M': 30,  'L': 30,  'A': 30}
}

# ==============================================================================
# HÀM TIỆN ÍCH CHUYỂN ĐỔI THỜI GIAN
# ==============================================================================
def time_to_sec(val):
    """Chuyển đổi chuỗi hh:mm hoặc hh:mm:ss sang giây."""
    if val is None or val == "" or str(val).strip() == "" or str(val).lower() == "nan":
        return None
    parts = list(map(int, str(val).strip().split(':')))
    if len(parts) == 2:
        return parts[0] * 3600 + parts[1] * 60
    elif len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    return 0

def sec_to_time(sec):
    """Chuyển đổi số giây sang định dạng hh:mm:ss."""
    if sec is None or sec == "":
        return ""
    sec = int(round(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

# ==============================================================================
# LỚP ĐỐI TƯỢNG CHUYẾN BAY (FLIGHT)
# ==============================================================================
class Flight:
    def __init__(self, id_code, f_type, category, terminal="", eobt_str="", etd_eta_str=""):
        self.id_code = str(id_code).strip()
        self.type = str(f_type).strip().upper()            # 'A' hoặc 'D'
        self.category = str(category).strip().upper()      # 'S', 'H', 'M', 'L'
        self.terminal = str(terminal).strip().upper() if terminal else ""
        
        self.eobt_str = str(eobt_str).strip() if eobt_str else ""
        self.etd_eta_str = str(etd_eta_str).strip() if etd_eta_str else ""
        
        self.EOBT = time_to_sec(self.eobt_str)
        self.ETD_ETA = time_to_sec(self.etd_eta_str)
        
        # Thiết lập các mốc thời gian đặc trưng
        if self.type == 'D':
            self.taxi_time = TERMINAL_TAXI_TIME.get(self.terminal, 360)
            if self.ETD_ETA is not None:
                self.ETD = self.ETD_ETA
            elif self.EOBT is not None:
                self.ETD = self.EOBT + self.taxi_time
            else:
                self.ETD = 0
            self.ETA = None
            self.E = self.ETD  # Mốc sẵn sàng cất cánh sớm nhất
        else: # 'A'
            self.taxi_time = CROSSING_TAXI_TIME
            self.ETA = self.ETD_ETA if self.ETD_ETA is not None else 0
            self.ETD = None
            self.EOBT = None
            # Máy bay hạ cánh tại ETA (mốc X), mất Y giây để tới điểm chờ cắt ngang
            self.E = self.ETA + CROSSING_TAXI_TIME
            
        self.O_i = 2       # Mặc định là chuyến bay độc lập
        self.R_i = 3       # Mặc định Medium
        self.J_i = 1 if IS_PEAK_HOUR else 2
        self.P_i = 1.0     # Hệ số ưu tiên
        self.mu = 1.0      # Hệ số phạt trễ
        
        # Biến quyết định khi xếp lịch
        self.t = None      # Thời gian thực tế sử dụng đường băng (ATD cho D, ATA/Crossing cho A)
        self.runway = None # Chỉ số đường băng được gán

# ==============================================================================
# TÍNH TOÁN THỨ BẬC ƯU TIÊN VÀ HỆ SỐ PHẠT (P_i và mu_i)
# ==============================================================================
def setup_flight_priorities(flights):
    """Xác định nối chuyến (O_i) và tính hệ số ưu tiên P_i, chi phí phạt mu_i."""
    dep_ids = {f.id_code for f in flights if f.type == 'D'}
    arr_ids = {f.id_code for f in flights if f.type == 'A'}
    connecting_ids = dep_ids.intersection(arr_ids)
    
    R_map = {'S': 1, 'H': 2, 'M': 3, 'L': 4}
    
    for f in flights:
        f.O_i = 1 if f.id_code in connecting_ids else 2
        f.R_i = R_map.get(f.category, 3)
        f.J_i = 1 if IS_PEAK_HOUR else 2
        
        PF = f.O_i + f.R_i + f.J_i
        term1 = (PF - 1) * (PF - 2) * (PF - 3) / 6.0
        term2 = (2 * PF - f.O_i - 2) * (f.O_i - 1) / 2.0
        f.P_i = term1 + term2 + f.J_i
        
        f.mu = 48.0 / f.P_i

# ==============================================================================
# KIỂM TRA RÀNG BUỘC VẬT LÝ & DUNG LƯỢNG HÀNG ĐỢI
# ==============================================================================
def check_queue_limit(candidate_t, flight, scheduled_flights):
    """Kiểm tra số lượng máy bay tại điểm chờ cắt ngang Arrival không vượt quá 2."""
    arrivals = [f for f in scheduled_flights if f.type == 'A']
    if flight.type == 'A':
        temp_f = copy.copy(flight)
        temp_f.t = candidate_t
        arrivals.append(temp_f)
        
    events = []
    for a in arrivals:
        events.append((a.E, 1))   # Đến điểm chờ cắt ngang (+1)
        events.append((a.t, -1))  # Rời điểm chờ cắt ngang qua đường băng (-1)
        
    events.sort(key=lambda x: (x[0], x[1]))
    
    q_size = 0
    for t_ev, diff in events:
        q_size += diff
        if q_size > 2:
            return False
    return True

def is_valid_insertion(runway_flights, flight, candidate_t):
    """Kiểm tra tính hợp lệ khi chèn chuyến bay vào mốc candidate_t trên đường băng."""
    if candidate_t < flight.E:
        return False
        
    for other in runway_flights:
        if candidate_t == other.t:
            return False
            
        # 1. Ràng buộc an toàn quanh mốc Arrival HẠ CÁNH (ETA):
        # - Chặn cất cánh trong 2 phút trước khi Arrival hạ cánh ([ETA - 120, ETA]).
        # - Sau khi Arrival hạ cánh, Departure phải tuân theo khoảng cách giãn cách wake turbulence.
        if flight.type == 'D' and other.type == 'A':
            sep_after_landing = SEPARATION_MATRIX[other.category][flight.category]
            if other.ETA - 120 <= candidate_t < other.ETA + sep_after_landing:
                return False
        elif flight.type == 'A' and other.type == 'D':
            sep_after_landing = SEPARATION_MATRIX[flight.category][other.category]
            if flight.ETA - 120 <= other.t < flight.ETA + sep_after_landing:
                return False
                
        # 2. Ràng buộc giãn cách bằng Separation Matrix trên đường băng (giữa t và candidate_t)
        # Hạng trên đường băng: D dùng category (S, H, M, L), A khi cắt ngang dùng 'A' (chặn 30s)
        cat_other = other.category if other.type == 'D' else 'A'
        cat_flight = flight.category if flight.type == 'D' else 'A'
        
        if other.t < candidate_t:
            req_sep = SEPARATION_MATRIX[cat_other][cat_flight]
            if candidate_t - other.t < req_sep:
                return False
        else:
            req_sep = SEPARATION_MATRIX[cat_flight][cat_other]
            if other.t - candidate_t < req_sep:
                return False
                
    # 3. Ràng buộc dung lượng điểm chờ Arrival không quá 2 máy bay
    if not check_queue_limit(candidate_t, flight, runway_flights):
        return False
        
    return True

def get_candidate_times(runway_flights, flight):
    """Sinh các mốc thời gian tiềm năng sớm nhất để chèn chuyến bay vào đường băng."""
    candidates = set()
    candidates.add(flight.E)
    
    cat_flight = flight.category if flight.type == 'D' else 'A'
    
    for other in runway_flights:
        cat_other = other.category if other.type == 'D' else 'A'
        
        # 1. Thời điểm ngay sau khi other sử dụng đường băng (cất cánh hoặc cắt ngang)
        earliest_after_other = other.t + SEPARATION_MATRIX[cat_other][cat_flight]
        candidates.add(earliest_after_other)
        
        # 2. Nếu other là Arrival và flight là Departure:
        # Thời điểm ngay sau khi other hạ cánh xong và thỏa mãn giãn cách wake turbulence
        if other.type == 'A' and flight.type == 'D':
            sep_landing = SEPARATION_MATRIX[other.category][flight.category]
            candidates.add(other.ETA + sep_landing)
            
    valid_candidates = []
    for ct in sorted(list(candidates)):
        if ct >= flight.E and is_valid_insertion(runway_flights, flight, ct):
            valid_candidates.append(ct)
    return valid_candidates

# ==============================================================================
# HÀM MỤC TIÊU ĐÁNH GIÁ ĐỘ TRỄ / CHI PHÍ (OBJECTIVE FUNCTIONS)
# ==============================================================================
def eval_obj1(flights):
    """Phương án 1: Tính tổng chi phí phạt của cả Departure và Arrival."""
    total_cost = 0.0
    for f in flights:
        if f.t is None: return float('inf')
        if f.type == 'D':
            delay = f.t - f.ETD
            total_cost += f.mu * delay
        else:
            # Độ trễ Arrival là thời gian phải chờ tại holding point (vượt quá 105s taxi)
            delay = max(0, f.t - f.E)
            total_cost += 1.0 * f.mu * delay
    return total_cost

def eval_obj2(flights):
    """Phương án 2: Phạt cực lớn cho thời gian trễ của máy bay Arrival (phi = 1000)."""
    total_cost = 0.0
    for f in flights:
        if f.t is None: return float('inf')
        if f.type == 'D':
            delay = f.t - f.ETD
            total_cost += f.mu * delay
        else:
            # phi = 1000 đối với Arrival (phạt cực lớn để ưu tiên giảm trễ Arrival tối đa)
            delay = max(0, f.t - f.E)
            total_cost += 1000.0 * f.mu * delay
    return total_cost

# ==============================================================================
# THUẬT TOÁN KHỞI TẠO THAM LAM (GREEDY HEURISTIC)
# ==============================================================================
def get_best_insertion(runways, flight, eval_func):
    best_cost_diff = float('inf')
    best_r = -1
    best_idx = -1
    best_t = -1
    
    for r in range(NUM_RUNWAYS):
        rw_flights = runways[r]
        valid_times = get_candidate_times(rw_flights, flight)
        
        for candidate_t in valid_times:
            idx = 0
            while idx < len(rw_flights) and rw_flights[idx].t < candidate_t:
                idx += 1
                
            if eval_func == eval_obj1:
                cost = (flight.mu * (candidate_t - flight.ETD)) if flight.type == 'D' else (1.0 * flight.mu * max(0, candidate_t - flight.E))
            else:
                cost = (flight.mu * (candidate_t - flight.ETD)) if flight.type == 'D' else (1000.0 * flight.mu * max(0, candidate_t - flight.E))
                
            if cost < best_cost_diff:
                best_cost_diff = cost
                best_r = r
                best_idx = idx
                best_t = candidate_t
                
    return best_r, best_idx, best_t

def greedy_schedule(flight_list, eval_func):
    """Xếp lịch tham lam tuần tự theo thứ tự thời gian dự kiến sẵn sàng E."""
    sorted_flights = sorted(flight_list, key=lambda f: f.E)
    runways = [[] for _ in range(NUM_RUNWAYS)]
    
    for f in sorted_flights:
        f_copy = copy.deepcopy(f)
        best_r, best_idx, best_t = get_best_insertion(runways, f_copy, eval_func)
        if best_r != -1:
            f_copy.t = best_t
            f_copy.runway = best_r
            runways[best_r].insert(best_idx, f_copy)
        else:
            # Nếu tạm thời nghẽn, đẩy lùi thời gian an toàn
            max_t = f_copy.E
            for r in range(NUM_RUNWAYS):
                if runways[r]:
                    max_t = max(max_t, runways[r][-1].t + 120)
            f_copy.t = max_t
            f_copy.runway = 0
            runways[0].append(f_copy)
            
    scheduled = []
    for r in range(NUM_RUNWAYS):
        scheduled.extend(runways[r])
    return scheduled, eval_func(scheduled)

# ==============================================================================
# THUẬT TOÁN SIMULATED ANNEALING & LARGE NEIGHBORHOOD SEARCH (SALNS)
# ==============================================================================
def salns_optimize(flight_list, eval_func, num_runs=NUM_SALNS_RUNS):
    """Chạy thuật toán SALNS num_runs lần và chọn kết quả có chi phí thấp nhất."""
    best_overall_sol = None
    best_overall_cost = float('inf')
    
    for run in range(num_runs):
        initial_sol, initial_cost = greedy_schedule(flight_list, eval_func)
        current_solution = copy.deepcopy(initial_sol)
        current_cost = initial_cost
        
        best_run_sol = copy.deepcopy(current_solution)
        best_run_cost = current_cost
        
        T = T_0
        mu = 0
        
        while T > T_L and mu < THETA_NOTIMP:
            improved_in_temp = False
            for _ in range(BETA):
                # 1. Phá hủy (Breaking / Destroy): tháo dỡ ngẫu nhiên 10% - 30% chuyến bay
                num_to_remove = max(1, int(len(current_solution) * random.uniform(0.1, 0.3)))
                remove_indices = random.sample(range(len(current_solution)), num_to_remove)
                
                removed_flights = [current_solution[i] for i in remove_indices]
                remaining_flights = [current_solution[i] for i in range(len(current_solution)) if i not in remove_indices]
                
                temp_runways = [[] for _ in range(NUM_RUNWAYS)]
                for f in remaining_flights:
                    temp_runways[f.runway].append(f)
                for r in range(NUM_RUNWAYS):
                    temp_runways[r].sort(key=lambda x: x.t)
                    
                # 2. Tái cấu trúc (Repair / Reorganization): chèn lại các chuyến bay
                random.shuffle(removed_flights)
                valid_repair = True
                
                for rf in removed_flights:
                    rf_clean = copy.deepcopy(rf)
                    rf_clean.t = None
                    rf_clean.runway = None
                    best_r, best_idx, best_t = get_best_insertion(temp_runways, rf_clean, eval_func)
                    
                    if best_r != -1:
                        rf_clean.t = best_t
                        rf_clean.runway = best_r
                        temp_runways[best_r].insert(best_idx, rf_clean)
                    else:
                        valid_repair = False
                        break
                        
                if valid_repair:
                    neighbor = []
                    for r in range(NUM_RUNWAYS):
                        neighbor.extend(temp_runways[r])
                    neighbor_cost = eval_func(neighbor)
                    
                    # 3. Tiêu chuẩn chấp nhận Metropolis
                    if neighbor_cost < current_cost:
                        current_solution = neighbor
                        current_cost = neighbor_cost
                        if neighbor_cost < best_run_cost:
                            best_run_sol = copy.deepcopy(neighbor)
                            best_run_cost = neighbor_cost
                            improved_in_temp = True
                    else:
                        try:
                            prob = math.exp(-(neighbor_cost - current_cost) / T)
                        except OverflowError:
                            prob = 0
                        if random.random() < prob:
                            current_solution = neighbor
                            current_cost = neighbor_cost
                            
            T *= LAMBDA
            if not improved_in_temp:
                mu += 1
            else:
                mu = 0
                
        if best_run_cost < best_overall_cost:
            best_overall_cost = best_run_cost
            best_overall_sol = copy.deepcopy(best_run_sol)
            
    return best_overall_sol, best_overall_cost

# ==============================================================================
# HÀM TÍNH TOÁN CHI TIẾT & ĐỊNH DẠNG BẢNG ĐẦU RA
# ==============================================================================
def process_output(sol, title, is_option_2=False):
    """Tính toán chi tiết độ trễ Gate, Holding, P, mu và in/chuẩn bị dữ liệu xuất."""
    sol_sorted = sorted(sol, key=lambda x: x.t)
    out_data = []
    delay_D_total = 0
    delay_A_total = 0
    total_cost_phi = 0.0
    
    for i, f in enumerate(sol_sorted):
        if f.type == 'D':
            delay = f.t - f.ETD
            delay_D_total += delay
            
            # Tính chiến lược lưu tại Gate tối đa 600s
            # AOBT = thời gian thực tế đẩy lùi
            # Để tiết kiệm xăng, giữ máy bay tại Gate tối đa 600s nếu chưa đến giờ cất cánh
            latest_pushback = f.t - f.taxi_time
            aobt_sec = max(f.EOBT, min(f.EOBT + MAX_GATE_HOLD_TIME, latest_pushback))
            gate_wait = aobt_sec - f.EOBT
            holding_wait = f.t - (aobt_sec + f.taxi_time)
            
            flight_cost = f.mu * delay
            total_cost_phi += flight_cost
            
            out_data.append({
                "STT": i + 1,
                "Mã chuyến bay": f.id_code,
                "Loại": f.type,
                "Hạng": f.category,
                "Nhà ga": f.terminal,
                "EOBT": sec_to_time(f.EOBT),
                "AOBT": sec_to_time(aobt_sec),
                "ETD/ETA": sec_to_time(f.ETD),
                "ATD/ATA": sec_to_time(f.t),
                "Chờ tại Gate": sec_to_time(gate_wait),
                "Chờ tại Holding": sec_to_time(holding_wait),
                "Thời gian Delay": sec_to_time(delay),
                "Hệ số P": round(f.P_i, 2),
                "Hệ số μ": round(f.mu, 3),
                "Chi phí trễ": round(flight_cost, 2)
            })
        else: # Arrival
            # Thời gian chờ tại điểm chờ cắt ngang (Holding point)
            holding_wait = max(0, f.t - f.E)
            # Thời gian thực tế hạ cánh / sử dụng đường băng ATA:
            # Nếu không bị nghẽn ở holding point thì ATA = ETA, ngược lại ATA bị trễ tương ứng
            ata_sec = f.ETA + holding_wait
            delay = holding_wait
            delay_A_total += delay
            
            if is_option_2:
                flight_cost = 1000.0 * f.mu * delay  # phi = 1000 đối với Arrival trong Phương án 2
            else:
                flight_cost = 1.0 * f.mu * delay
                total_cost_phi += flight_cost
                
            out_data.append({
                "STT": i + 1,
                "Mã chuyến bay": f.id_code,
                "Loại": f.type,
                "Hạng": f.category,
                "Nhà ga": "",
                "EOBT": "",
                "AOBT": "",
                "ETD/ETA": sec_to_time(f.ETA),
                "ATD/ATA": sec_to_time(f.t),
                "Chờ tại Gate": "",
                "Chờ tại Holding": sec_to_time(holding_wait),
                "Thời gian Delay": sec_to_time(delay),
                "Hệ số P": round(f.P_i, 2),
                "Hệ số μ": round(f.mu, 3),
                "Chi phí trễ": round(flight_cost, 2)
            })
            
    if USE_EXCEL == 0:
        print(f"\n{'='*115}")
        print(f"{title}")
        print(f"{'='*115}")
        header = f"{'STT':<4} | {'Mã CB':<7} | {'Loại':<4} | {'Hạng':<4} | {'Ga':<3} | {'EOBT':<8} | {'AOBT':<8} | {'ETD/ETA':<8} | {'ATD/ATA':<8} | {'Chờ Gate':<8} | {'Chờ Hold':<8} | {'Delay':<8} | {'P':<5} | {'μ':<5} | {'Chi phí':<8}"
        print(header)
        print("-" * 115)
        for r in out_data:
            line = f"{r['STT']:<4} | {r['Mã chuyến bay']:<7} | {r['Loại']:<4} | {r['Hạng']:<4} | {r['Nhà ga']:<3} | {r['EOBT']:<8} | {r['AOBT']:<8} | {r['ETD/ETA']:<8} | {r['ATD/ATA']:<8} | {r['Chờ tại Gate']:<8} | {r['Chờ tại Holding']:<8} | {r['Thời gian Delay']:<8} | {r['Hệ số P']:<5} | {r['Hệ số μ']:<5} | {r['Chi phí trễ']:<8}"
            print(line)
        print("-" * 115)
        print(f"Tổng trễ Departure : {sec_to_time(delay_D_total)} ({delay_D_total}s)")
        print(f"Tổng trễ Arrival   : {sec_to_time(delay_A_total)} ({delay_A_total}s)")
        print(f"Tổng Chi phí Hàm Mục Tiêu Φ: {round(total_cost_phi, 2)}")
        print(f"{'='*115}\n")
        
    return out_data, delay_D_total, delay_A_total, total_cost_phi

# ==============================================================================
# HÀM CHÍNH (MAIN)
# ==============================================================================
def main():
    if USE_EXCEL == 1:
        # Kiểm tra file output có đang bị khóa bởi Excel không
        if os.path.exists(OUTPUT_EXCEL_FILE):
            try:
                os.remove(OUTPUT_EXCEL_FILE)
            except PermissionError:
                print(f"\n[LỖI]: File '{OUTPUT_EXCEL_FILE}' đang được mở bởi Microsoft Excel hoặc ứng dụng khác.")
                print("Vui lòng đóng file đó lại trước khi chạy chương trình!\n")
                return

        try:
            import pandas as pd
        except ImportError:
            print("Lỗi: Không tìm thấy thư viện pandas. Hãy chạy lệnh 'pip install pandas openpyxl' để cài đặt.")
            return

        try:
            df = pd.read_excel(INPUT_EXCEL_FILE)
            flights = []
            for _, row in df.iterrows():
                id_code = str(row['CALL SIGN'])
                f_type = str(row['TYPE A/C'])
                cat = str(row.get('WAKE TURBULENCE', 'M'))
                terminal = str(row.get('TERMINAL', '')) if f_type.upper() == 'D' else ""
                eobt = str(row.get('EOBT', '')) if f_type.upper() == 'D' else ""
                etd_eta = str(row.get('ETD/ETA', ''))
                flights.append(Flight(id_code, f_type, cat, terminal, eobt, etd_eta))
        except Exception as e:
            print(f"Lỗi đọc file Excel {INPUT_EXCEL_FILE}: {e}")
            return
    else:
        # Dữ liệu mẫu thực tế 12 chuyến bay tập trung trong khung giờ cao điểm có xung đột
        # VN202 xuất hiện ở cả Arrival và Departure -> Chuyến bay nối chuyến (O_i = 1)
        # Các chuyến bay có giờ dự kiến sát nhau để kiểm tra thuật toán tối ưu hóa
        flights = [
            Flight("VJ101", "A", "M", "",   "",      "01:00"),
            Flight("BA001", "D", "S", "T1", "00:55", "01:01"),  # Super, T1 (taxi 6m)
            Flight("VN202", "A", "H", "",   "",      "01:02"),  # Nối chuyến (O_i = 1), Heavy
            Flight("VN202", "D", "H", "T2", "00:58", "01:02"),  # Nối chuyến (O_i = 1), Heavy, T2 (taxi 4m)
            Flight("QH303", "A", "L", "",   "",      "01:05"),  # Light
            Flight("VJ404", "D", "M", "T3", "00:57", "01:05"),  # Medium, T3 (taxi 8m)
            Flight("VN505", "A", "M", "",   "",      "01:08"),
            Flight("QH606", "D", "L", "T1", "01:03", "01:09"),  # Light, T1 (taxi 6m)
            Flight("AF707", "D", "S", "T2", "01:06", "01:10"),  # Super, T2 (taxi 4m)
            Flight("VJ808", "A", "H", "",   "",      "01:12"),
            Flight("VN909", "D", "M", "T3", "01:05", "01:13"),  # Medium, T3 (taxi 8m)
            Flight("QH010", "D", "H", "T1", "01:08", "01:14")   # Heavy, T1 (taxi 6m)
        ]

    # Tính toán các hệ số ưu tiên P_i và mu_i
    setup_flight_priorities(flights)
    
    print("=" * 70)
    print(" HỆ THỐNG XẾP LỊCH ĐƯỜNG BĂNG SALNS V2 (ICAO & RESEARCH MODEL)")
    print(f" Chế độ Giờ cao điểm: {'BẬT (J=1)' if IS_PEAK_HOUR else 'TẮT (J=2)'}")
    print(f" Số lần chạy tối ưu SALNS: {NUM_SALNS_RUNS}")
    print("=" * 70)
    print("Đang tiến hành tối ưu hóa lịch trình, vui lòng đợi trong giây lát...\n")

    # --------------------------------------------------------------------------
    # PHƯƠNG ÁN 1: TỐI ƯU TỔNG CHI PHÍ PHẠT (CẢ DEPARTURE & ARRIVAL)
    # --------------------------------------------------------------------------
    greedy_sol1, _ = greedy_schedule(flights, eval_obj1)
    salns_sol1, _ = salns_optimize(flights, eval_obj1, num_runs=NUM_SALNS_RUNS)

    # --------------------------------------------------------------------------
    # PHƯƠNG ÁN 2: CHỈ TỐI ƯU CHI PHÍ PHẠT DEPARTURE (ARRIVAL PHI = 0)
    # --------------------------------------------------------------------------
    greedy_sol2, _ = greedy_schedule(flights, eval_obj2)
    salns_sol2, _ = salns_optimize(flights, eval_obj2, num_runs=NUM_SALNS_RUNS)

    # Xuất kết quả
    data_g1, d_D_g1, d_A_g1, cost_g1 = process_output(greedy_sol1, ">>> [PHƯƠNG ÁN 1] KẾT QUẢ GREEDY BASELINE (CẢ D & A)", is_option_2=False)
    data_s1, d_D_s1, d_A_s1, cost_s1 = process_output(salns_sol1, ">>> [PHƯƠNG ÁN 1] KẾT QUẢ TỐI ƯU SALNS (CẢ D & A)", is_option_2=False)

    data_g2, d_D_g2, d_A_g2, cost_g2 = process_output(greedy_sol2, ">>> [PHƯƠNG ÁN 2] KẾT QUẢ GREEDY BASELINE (ƯU TIÊN TUYỆT ĐỐI ARRIVAL - PHI_A = 1000)", is_option_2=True)
    data_s2, d_D_s2, d_A_s2, cost_s2 = process_output(salns_sol2, ">>> [PHƯƠNG ÁN 2] KẾT QUẢ TỐI ƯU SALNS (ƯU TIÊN TUYỆT ĐỐI ARRIVAL - PHI_A = 1000)", is_option_2=True)

    if USE_EXCEL == 1:
        import pandas as pd
        with pd.ExcelWriter(OUTPUT_EXCEL_FILE) as writer:
            def write_sheet(data, d_D, d_A, total_cost, sheet_name):
                df_out = pd.DataFrame(data)
                df_out.to_excel(writer, sheet_name=sheet_name, index=False)
                worksheet = writer.sheets[sheet_name]
                last_row = len(data) + 2
                worksheet.cell(row=last_row, column=1, value="Tổng trễ D:")
                worksheet.cell(row=last_row, column=2, value=sec_to_time(d_D))
                worksheet.cell(row=last_row + 1, column=1, value="Tổng trễ A:")
                worksheet.cell(row=last_row + 1, column=2, value=sec_to_time(d_A))
                worksheet.cell(row=last_row + 2, column=1, value="Tổng Chi phí Φ:")
                worksheet.cell(row=last_row + 2, column=2, value=round(total_cost, 2))

            write_sheet(data_g1, d_D_g1, d_A_g1, cost_g1, "PA1_Greedy_Baseline")
            write_sheet(data_s1, d_D_s1, d_A_s1, cost_s1, "PA1_SALNS_Toi_Uu")
            write_sheet(data_g2, d_D_g2, d_A_g2, cost_g2, "PA2_Greedy_Baseline")
            write_sheet(data_s2, d_D_s2, d_A_s2, cost_s2, "PA2_SALNS_Toi_Uu")

        print(f"\n[THÀNH CÔNG]: Đã xuất kết quả cả 2 phương án ra file Excel '{OUTPUT_EXCEL_FILE}' (4 Sheets).")

if __name__ == "__main__":
    main()


 HỆ THỐNG XẾP LỊCH ĐƯỜNG BĂNG SALNS V2 (ICAO & RESEARCH MODEL)
 Chế độ Giờ cao điểm: BẬT (J=1)
 Số lần chạy tối ưu SALNS: 10
Đang tiến hành tối ưu hóa lịch trình, vui lòng đợi trong giây lát...


[THÀNH CÔNG]: Đã xuất kết quả cả 2 phương án ra file Excel 'output_schedule.xlsx' (4 Sheets).
